# [5.2] Gemma Scope and Feature Steering - Solutions

These cells validate the reference implementations in `solutions.py`. The learner notebook asks you to implement the same functions directly; this notebook is intentionally short, because the source file is the single reference implementation used by the report generator and tests.

<img src="../../instructions/assets/gemma_scope_validation_ladder.svg" width="820">


In [ ]:
import json
import sys
from pathlib import Path

chapter = "chapter5_modern_architectures"
section = "part2_gemma_scope_feature_steering"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

from part2_gemma_scope_feature_steering import solutions, tests


## 1. Sparsity Metrics

<details><summary>Expected output</summary>

```text
All tests in `test_feature_density_l0_and_dead_fraction_match_reference` passed!
```

</details>

<details><summary>Help - solution idea</summary>

The implementation builds a boolean firing mask, averages across every non-feature axis for density, sums across the feature axis for L0, and counts features with zero firing rate for dead-feature fraction.

</details>


In [ ]:
tests.test_feature_density_l0_and_dead_fraction_match_reference(
    solutions.feature_density,
    solutions.l0,
    solutions.dead_feature_fraction,
)


## 2. Reconstruction Metrics

<details><summary>Expected output</summary>

```text
All tests in `test_compute_sae_reconstruction_metrics_matches_manual_and_reference` passed!
```

</details>

<details><summary>Help - solution idea</summary>

The reference implementation validates paired optional arguments, computes MSE in float precision, computes KL on log-softmax probabilities, and only reports loss recovered when all three loss values are present.

</details>


In [ ]:
tests.test_compute_sae_reconstruction_metrics_matches_manual_and_reference(
    solutions.compute_sae_reconstruction_metrics,
)


## 3. Direct Logit Attribution

<details><summary>Expected output</summary>

```text
All tests in `test_direct_logit_attribution_projects_decoder_vectors` passed!
```

</details>

<details><summary>Help - solution idea</summary>

Project decoder vectors through the unembedding with matrix multiplication, then select requested token columns if `token_ids` is provided. Shape checks make silent transposes fail loudly.

</details>


In [ ]:
tests.test_direct_logit_attribution_projects_decoder_vectors(
    solutions.direct_logit_attribution,
)


## 4. Held-Out Feature Validation

<details><summary>Expected output</summary>

```text
All tests in `test_feature_detection_report_handles_heldout_controls` passed!
```

</details>

<details><summary>Help - solution idea</summary>

The tie-aware AUC uses average ranks. The default threshold is the midpoint between positive and negative means, which is simple enough to inspect and deterministic enough for a course test.

</details>


In [ ]:
tests.test_feature_detection_report_handles_heldout_controls(
    solutions.feature_detection_report,
    solutions.roc_auc_binary,
)


## 5. Top Activations and Ablation

<details><summary>Expected output</summary>

```text
All tests in `test_top_activating_examples_and_ablation_match_reference` passed!
```

</details>

<details><summary>Help - solution idea</summary>

`top_activating_examples` flattens all non-feature axes before `topk`. `ablate_features` clones the activation tensor and replaces selected feature columns with either zero or their global mean.

</details>


In [ ]:
tests.test_top_activating_examples_and_ablation_match_reference(
    solutions.top_activating_examples,
    solutions.ablate_features,
)


## 6. Decoder Steering Controls

<details><summary>Expected output</summary>

```text
All tests in `test_decoder_steering_and_random_control_report` passed!
```

</details>

<details><summary>Help - solution idea</summary>

The selected decoder vectors are weighted, summed, and added either to every token position or only the last one. The report compares the target-score delta against the absolute random-control delta.

</details>


In [ ]:
tests.test_decoder_steering_and_random_control_report(
    solutions.apply_decoder_steering,
    solutions.steering_comparison_report,
)


## 7. Whole-Notebook Contract

<details><summary>Expected output</summary>

```text
All tests in `test_notebook_contract` passed!
```

</details>

<details><summary>Help - what this validates</summary>

This smoke path confirms that the public functions compose into one deterministic result with nonzero sparse activity, perfect held-out toy separation, correct DLA values, a steering control pass, and feature ablation.

</details>


In [ ]:
tests.test_notebook_contract(solutions.run_smoke_test)


## Signature Result

| Check | Required result |
|---|---:|
| Gemma Scope artifact preflight | `true` |
| Gemma Scope forward pass | `true` |
| SAE width | `16384` |
| Residual width | `1152` |
| Held-out feature AUC | `1.000` |
| Random-feature baseline AUC | `0.500` |
| Label-shuffle control | `true` |
| Peak VRAM | `< 24 GB` |

<details><summary>Interpreting the signature result</summary>

The reference implementation is accepted only when the notebook contract, CUDA artifact preflight, and real-activation feature validation are all recorded in `verification_report.json`. The result is a narrow, evidence-backed feature validation claim, not a blanket interpretability claim about Gemma Scope.

</details>

<details><summary>Help - checking the committed report</summary>

The cell below reads the committed report rather than rerunning the gated model path. Use the report generator when you need to refresh GPU evidence after source changes.

</details>


In [ ]:
def _load_committed_gpu_report() -> dict:
    report_path = section_dir / "verification_report.json"
    report = json.loads(report_path.read_text())
    gpu = report["metrics"]["gpu_test"]
    return {
        "accepted": report["accepted"],
        "tests": report["tests_passed"],
        "device": gpu["device"],
        "torch": gpu["torch_version"],
        "cuda": gpu["cuda_version"],
        "artifact_preflight": gpu["gemma_scope_artifact_preflight_passed"],
        "forward_pass": gpu["gemma_scope_forward_passed"],
        "feature_auc": gpu["gemma_scope_real_activation_feature_auc"],
        "baseline_auc": gpu["gemma_scope_real_activation_baseline_auc"],
        "label_shuffle_control": gpu["gemma_scope_real_activation_label_shuffle_control_passed"],
        "peak_vram_gb": gpu["peak_vram_gb"],
    }

_load_committed_gpu_report()


## Limitations

The reference solution validates the local feature-steering ladder and the pinned Gemma Scope artifact path. It does not provide a complete taxonomy of Gemma Scope features, certify a steering vector as safe, or replace broader mechanistic audits on larger held-out datasets.
